# **Setup Environment**

In [1]:
import sys
import os
import numpy as np
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))

from gymnasium.vector import SyncVectorEnv
from core.env.core import SnakeEnv
from core.env.types import ObserveType, RewardOptions

num_envs, total_episodes = 16, 100000
env = SyncVectorEnv(
    [
        lambda i=i: SnakeEnv(
            width=20,
            height=20,
            obs_type=ObserveType.VEC_11,
            num_apples=3,
            num_obstacles=15,
            seed=42 + i,
            reward_options=RewardOptions(
                eats_apple=24.0,
                penalty_step=-0.01,
                penalty_loop=-0.1,
                death_wall=-20.0,
                death_self=-20.0,
                shaping_closer=0.1,
                shaping_further=-0.1,
                complete=100.0,
            ),
        )
        for i in range(num_envs)
    ]
)

epsilon_decay = (0.01) ** (1 / (total_episodes * 0.6))

training_logs = []
episode_rewards = np.zeros(num_envs)
completed = 0
states, infos = env.reset()
best_reward = -np.inf

---

# **Initialize Agent**

In [2]:
from agents.q_learning import QLearningAgent

agent_name = "q_learning_snake_2"
agent = QLearningAgent(
    state_dim=2048,
    action_dim=3,
    lr=0.05,
    gamma=0.99,
    epsilon_decay=epsilon_decay,
    seed=42,
)

---

# **Train Agent**

In [3]:
with tqdm(total=total_episodes, desc="Parallel Training") as pbar:
    while completed < total_episodes:
        actions = [int(agent.act(s)) for s in states]
        next_states, rewards, terminated, truncated, next_infos = env.step(actions)
        for i in range(len(actions)):
            s = states[i]
            a = actions[i]
            r = float(rewards[i])
            ns = next_states[i]
            done_i = bool(terminated[i])
            trunc_i = bool(truncated[i])

            agent.update(s, a, r, ns, done_i)
            episode_rewards[i] += r

            if done_i or trunc_i:
                completed += 1
                if completed <= total_episodes:
                    pbar.update(1)
                    agent.train()
                    training_logs.append(
                        {
                            "episode": completed,
                            "reward": float(episode_rewards[i]),
                            "epsilon": agent.epsilon,
                        }
                    )

                    if len(training_logs) >= 100:
                        recent_avg = np.mean([log["reward"] for log in training_logs[-100:]])
                        if recent_avg > best_reward:
                            best_reward = recent_avg
                            agent.save(f"{agent_name}_best.pkl")

                    if completed % 5000 == 0:
                        recent_avg = (
                            np.mean([log["reward"] for log in training_logs[-100:]])
                            if len(training_logs) >= 100
                            else float(episode_rewards[i])
                        )
                        tqdm.write(
                            f"Ep {completed}/{total_episodes} | Avg Reward (last 100): {recent_avg:.2f} | Eps: {agent.epsilon:.3f} | Best Avg: {best_reward:.2f}"
                        )
                episode_rewards[i] = 0.0

        states = next_states

env.close()

Parallel Training:   5%|▌         | 5183/100000 [00:04<01:21, 1168.35it/s]

Ep 5000/100000 | Avg Reward (last 100): -8.01 | Eps: 0.681 | Best Avg: -4.79


Parallel Training:  10%|█         | 10070/100000 [00:09<01:45, 852.90it/s] 

Ep 10000/100000 | Avg Reward (last 100): 1.15 | Eps: 0.464 | Best Avg: 7.42


Parallel Training:  15%|█▌        | 15106/100000 [00:14<01:25, 992.21it/s] 

...

Ep 90000/100000 | Avg Reward (last 100): 392.01 | Eps: 0.010 | Best Avg: 473.79


Parallel Training:  95%|█████████▌| 95019/100000 [06:50<00:29, 167.18it/s]

Ep 95000/100000 | Avg Reward (last 100): 322.14 | Eps: 0.010 | Best Avg: 473.79


Parallel Training: 100%|██████████| 100000/100000 [07:23<00:00, 225.49it/s]

Ep 100000/100000 | Avg Reward (last 100): 371.44 | Eps: 0.010 | Best Avg: 481.33


## **Save Last Model**

In [4]:
agent.save(f"{agent_name}.pkl")

---

# **Evaluate Agent**

In [5]:
from core.utils import save_metrics

save_metrics(training_logs, f"{agent_name}_training_logs.csv")

In [6]:
from core.utils import evaluate_agent

evaluate_agent(agent, seed=67)

Evaluating Agent: 100%|██████████| 100/100 [00:01<00:00, 91.33it/s]


Evaluation Metrics                                        
╭─────────┬─────────┬────────┬────────┬────────┬─────────╮
│ Metric  │ Average │ Median │    Min │    Max │ Std Dev │
├─────────┼─────────┼────────┼────────┼────────┼─────────┤
│ Rewards │  214.77 │ 205.40 │ -56.60 │ 642.00 │  131.57 │
│ Apples  │   17.77 │  17.00 │   0.00 │  49.00 │    9.60 │
│ Steps   │  281.13 │ 257.00 │   2.00 │ 876.00 │  156.86 │
╰─────────┴─────────┴────────┴────────┴────────┴─────────╯

Death Distribution             
╭────────┬───────┬────────────╮
│ Reason │ Count │ Percentage │
├────────┼───────┼────────────┤
│ Self   │    77 │      77.0% │
│ Wall   │    23 │      23.0% │
╰────────┴───────┴────────────╯

({'Average': 214.76799999999955,
  'Median': 205.4000000000008,
  'Min': -56.59999999999912,
  'Max': 641.9999999999897,
  'Std Dev': 131.56529244447265},
 {'Average': 17.77,
  'Median': 17.0,
  'Min': 0.0,
  'Max': 49.0,
  'Std Dev': 9.602973497828682},
 {'Average': 281.13,
  'Median': 257.0,
  'Min': 2.0,
  'Max': 876.0,
  'Std Dev': 156.85800298359024},
 {<DeathReason.WALL: 'Wall'>: 23, <DeathReason.SELF: 'Self'>: 77})

---